### Neural Network For Language Translation

In [36]:
#importing libraries

import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import unicodedata
import re
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Layer, Input, Masking, LSTM, Embedding, Dense
from tensorflow.keras.models import Model


In [2]:
# loading dataset

NUM_EXAMPLES = 20000
data_examples = []
with open("data/deu.txt", "r", encoding="utf-8") as data:
    for line in data.readlines():
        if len(data_examples) < 20000:
            data_examples.append(line)

In [3]:
#sample data
data_examples[0]

'Hi.\tHallo!\tCC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #380701 (cburgmer)\n'

In [4]:
#separating english and german sentences into different lists
english_list = []
german_list = []
for i in range(len(data_examples)):
    english_sent, german_sent = data_examples[i].split("\t")[:2]
    english_list.append(english_sent)
    german_list.append(german_sent)


In [5]:
#some random english and german sentences

inx = np.random.choice(len(data_examples), 5, replace=True)
for i in inx:
    print(f"English sentence: {english_list[i]}")
    print(f"German sentence: {german_list[i]}")
    print()

English sentence: Tom exaggerated.
German sentence: Tom übertrieb.

English sentence: Why do we sneeze?
German sentence: Warum niesen wir?

English sentence: They're bad.
German sentence: Sie sind schlecht.

English sentence: I got angry.
German sentence: Ich wurde wütend.

English sentence: Now you're safe.
German sentence: Jetzt sind Sie in Sicherheit.



In [6]:
# These functions preprocess English and German sentences

def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r"ü", 'ue', sentence)
    sentence = re.sub(r"ä", 'ae', sentence)
    sentence = re.sub(r"ö", 'oe', sentence)
    sentence = re.sub(r'ß', 'ss', sentence)
    
    sentence = unicode_to_ascii(sentence)
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r"[^a-z?.!,']+", " ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    
    return sentence.strip()

In [7]:
preprocessed_german = list(map(preprocess_sentence, german_list))
preprocessed_english = list(map(preprocess_sentence, english_list))

In [8]:
#adding <start> and <end> tokens for greman list of sentences

token_german=[]
for i in range(len(preprocessed_german)):
    token_german.append("<start> " + preprocessed_german[i] + " <end>")
    
print(token_german[0]+"\n") 

print(preprocessed_german[0])

<start> hallo ! <end>

hallo !


In [9]:
#tokenizing german sentences


tokenizer = Tokenizer(lower=True, char_level=False)

tokenizer.fit_on_texts(token_german)
german_sequence = tokenizer.texts_to_sequences(token_german)

In [10]:
vocabulary = tokenizer.word_index

In [11]:
#samples view

inx = np.random.choice(len(token_german), 5, replace=False)

for i in inx:
    print(preprocessed_english[i])
    print(preprocessed_german[i])
    print(german_sequence[i])
    print(token_german[i])
    print()

he is outgoing .
er ist aufgeschlossen .
[1, 11, 5, 1643, 2]
<start> er ist aufgeschlossen . <end>

you may have it .
du kannst es haben .
[1, 10, 105, 7, 31, 2]
<start> du kannst es haben . <end>

i can't stay .
ich kann nicht bleiben .
[1, 3, 26, 9, 218, 2]
<start> ich kann nicht bleiben . <end>

he's in danger .
er ist in gefahr .
[1, 11, 5, 42, 852, 2]
<start> er ist in gefahr . <end>

tom is spirited .
tom ist temperamentvoll .
[1, 4, 5, 4559, 2]
<start> tom ist temperamentvoll . <end>



In [12]:
#padding german sequences

padded_german_sequences = pad_sequences(german_sequence, padding="post")

In [13]:
#embedding english list
#load embedding module from Tensorflow Hub

embedding_layer = hub.KerasLayer("https://tfhub.dev/google/tf2-preview/nnlm-en-dim128/1", 
                                 output_shape=[128], input_shape=[], dtype=tf.string)

In [14]:
#testing the layer

embedding_layer(tf.constant(["these", "aren't", "the", "droids", "you're", "looking", "for"])).shape

TensorShape([7, 128])

NOTE:

- As part of data preprocessing we will use a pre-trained English word embedding module from TensorFlow Hub.
- This embedding takes a batch of text tokens in a 1-D tensor of strings as input. It then embeds the separate tokens into a 128-dimensional space.
- This model can also be used as a sentence embedding module. The module will process each token by removing punctuation and splitting on spaces. It then averages the word embeddings over a sentence to give a single embedding vector. However, we will use it only as a word embedding module, and will pass each word in the input sentence as a separate token.


In [15]:
#splitting data into training and validation sets

training_len= int(len(preprocessed_english) - 0.2*(len(preprocessed_english)))

train_english = preprocessed_english[: training_len]
train_german = padded_german_sequences[: training_len]

validation_english = preprocessed_english[training_len:]
validation_german = padded_german_sequences[training_len:]

In [16]:
#loading them into dataset

train_dataset = tf.data.Dataset.from_tensor_slices((train_english, train_german))
validation_dataset = tf.data.Dataset.from_tensor_slices((validation_english, validation_german))

In [17]:
#splitting english sentences into words to embed them

train_dataset = train_dataset.map(lambda english, german : (tf.strings.split(english, sep=" "), german))
validation_dataset = validation_dataset.map(lambda english, german : (tf.strings.split(english, sep=" "), german))

In [18]:
#passing english words through embedding layer created earlier

train_dataset = train_dataset.map(lambda english, german : (embedding_layer(english), german))
validation_dataset = validation_dataset.map(lambda english, german : (embedding_layer(english), german))

In [19]:
#filtering out dataset items if words in english sentence is greater than 13.
train_dataset = train_dataset.filter(lambda english, german : tf.less(tf.shape(english)[0], 13))
validation_dataset = validation_dataset.filter(lambda english, german : tf.less(tf.shape(english)[0], 13))

In [20]:
#adding padding to each english sequence vector

def adjust_dims(english, german):
    h = tf.shape(english)[0]
    english, german = tf.pad(english, [[0, 13-h], [0, 0]]), german
    return english, german

train_dataset = train_dataset.map(adjust_dims)
validation_dataset = validation_dataset.map(adjust_dims)

In [21]:
#batch the dataset items

train_dataset = train_dataset.batch(16)
validation_dataset = validation_dataset.batch(16)


In [22]:
#element spec of train_dataset

train_dataset.element_spec

(TensorSpec(shape=(None, None, 128), dtype=tf.float32, name=None),
 TensorSpec(shape=(None, 12), dtype=tf.int32, name=None))

In [23]:
#element spec of validation_dataset

validation_dataset.element_spec

(TensorSpec(shape=(None, None, 128), dtype=tf.float32, name=None),
 TensorSpec(shape=(None, 12), dtype=tf.int32, name=None))

In [24]:
#shape of english data example from training dataset

[x[0].shape for x in train_dataset.take(1)]

[TensorShape([16, 13, 128])]

In [25]:
#shape of german data example from validation dataset

[x[1].shape for x in validation_dataset.take(1)]

[TensorShape([16, 12])]

In [26]:
#creation of custom layer

class MyLayer(Layer):
    
    def __init__(self, **kwargs):
        super(MyLayer, self).__init__( **kwargs)
        self.w = tf.Variable(initial_value=tf.random.normal((128,)), trainable=True, name="end_token")
        
    def call(self, inputs):
        batch_size=tf.shape(inputs)[0]
        seq_len= tf.shape(inputs)[1]
        w_expanded=tf.reshape(self.w, (1,1,128))
        w_expanded = tf.tile(input=w_expanded, multiples=[batch_size, 1, 1])
        outputs= tf.concat([inputs, w_expanded], axis=1)
        return outputs

In [27]:
#testing layer with random input
sample_shape=[x[0].shape for x in train_dataset.take(1)][0]
mylayer = MyLayer()
mylayer(tf.zeros(sample_shape))

<tf.Tensor: shape=(16, 14, 128), dtype=float32, numpy=
array([[[ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        ...,
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 1.4915885 , -0.5272867 ,  0.46548933, ..., -2.59096   ,
         -0.8271714 , -1.8682442 ]],

       [[ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        ...,

In [28]:
#building encoder model using functional api

inputs = Input(shape=(None, 128))
x = MyLayer()(inputs)
x = Masking(mask_value=0)(x)
lstm_output, state_h, state_c = LSTM(units=512, return_sequences=True, return_state=True)(x)

encoder_model = Model(inputs=inputs, outputs=[state_h, state_c])
encoder_model.summary()

encoder_states = [state_h, state_c]

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None, 128) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ my_layer_1          │ (None, None, 128) │          0 │ input_layer[0][0] │
│ (MyLayer)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None, 128) │          0 │ my_layer_1[0][0]  │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking (Masking)   │ (None, None, 128) │          0 │ my_layer_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any (Any)           │ (None, None)      │          0 │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, None,     │  1,312,768 │ masking[0][0],    │
│                     │ 512), (None,      │            │ any[0][0]         │
│                     │ 512), (None,      │            │                   │
│                     │ 512)]             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,312,768 (5.01 MB)

 Trainable params: 1,312,768 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

In [29]:
#buiding decoder model using sub-classing api

vocab_size=max(tokenizer.word_index.values())

class MyDecoder(Model):
    
    def __init__(self, **kwargs):
        
        super(MyDecoder, self).__init__(**kwargs)
        
        self.embedding = Embedding(input_dim=5739, output_dim=128, mask_zero=True)
        self.lstm = LSTM(units=512, return_state=True, return_sequences=True)
        self.dense = Dense(units=5739)
        
    def call(self, inputs, encoder_states):
        x = self.embedding(inputs)
        x, dstate_h , dstate_c = self.lstm(x, initial_state=encoder_states)
        outputs = self.dense(x)
        return outputs, dstate_h, dstate_c
        

In [30]:
#testing encoder and decoder model with sample data

encoder_english_in=[x for x in train_dataset.take(1)][0][0]
decoder_german_in=[x for x in train_dataset.take(1)][0][1]

decoder_model = MyDecoder()
decoder_model(inputs=decoder_german_in, encoder_states=encoder_model(encoder_english_in))

(<tf.Tensor: shape=(16, 12, 5739), dtype=float32, numpy=
 array([[[ 2.53020041e-02, -2.84244604e-02, -5.30840550e-03, ...,
           3.79468780e-03,  7.42947496e-03,  1.83467921e-02],
         [ 1.78951975e-02, -1.88362002e-02, -8.70633684e-03, ...,
           5.26245777e-03,  1.85637875e-03,  9.69186146e-03],
         [ 1.36943292e-02, -9.97303613e-03, -9.39372275e-03, ...,
           4.41629905e-03, -1.03348715e-03,  5.64754196e-03],
         ...,
         [ 1.08133545e-02,  9.33704525e-03, -7.11997505e-03, ...,
           1.08003663e-03, -7.82800745e-03, -1.90028150e-05],
         [ 1.06446836e-02,  9.57194064e-03, -6.44236477e-03, ...,
           6.67975168e-04, -7.54094822e-03, -1.12728245e-04],
         [ 1.04870694e-02,  9.62327328e-03, -5.82383247e-03, ...,
           3.37596110e-04, -7.21112918e-03, -2.12374536e-04]],
 
        [[ 2.53020041e-02, -2.84244604e-02, -5.30840550e-03, ...,
           3.79468780e-03,  7.42947496e-03,  1.83467921e-02],
         [ 1.98651534e-02, -1.

In [31]:
decoder_model.summary()

Model: "my_decoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (16, 12, 128)          │       734,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ((16, 12, 512), (16,   │     1,312,768 │
│                                 │ 512), (16, 512))       │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (16, 12, 5739)         │     2,944,107 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,991,467 (19.04 MB)

 Trainable params: 4,991,467 (19.04 MB)

 Non-trainable params: 0 (0.00 B)

In [32]:
#custom training loop

def prepare_decoder_data(english_batch, german_batch):
    
    decoder_input = german_batch[:, :-1]
    decoder_output = german_batch[:, 1:]
    
    return  (decoder_input, decoder_output)



In [33]:
#defining optimizer and loss function

optimizer_obj = tf.keras.optimizers.Adam(learning_rate=0.001)
loss_obj = tf.keras.losses.SparseCategoricalCrossentropy()

In [34]:
@tf.function
def forward_and_backward_prop(english_batch, decoder_input, decoder_output):
    with tf.GradientTape() as tape:
        encoder_hidden_state, encoder_cell_state = encoder_model(english_batch)
        dout, _, _=decoder_model(inputs=decoder_input, encoder_states=(encoder_hidden_state, encoder_cell_state))
        loss_value = loss_obj(decoder_output, dout)
        grads = tape.gradient(loss_value, encoder_model.trainable_variables + decoder_model.trainable_variables)
    return loss_value, grads
    

In [37]:
def training_loop(train_dataset, test_dataset, epochs=5):
    
    training_loss = []
    validation_loss = []
    for epoch in range(epochs):
        batch_losses = []
        validation_losses = []
        
        for english_batch, german_batch in train_dataset:
            decoder_input, decoder_output = prepare_decoder_data(english_batch, german_batch)
            loss_value, grads = forward_and_backward_prop(english_batch, decoder_input, decoder_output)
            batch_losses.append(loss_value.numpy())
            optimizer_obj.apply_gradients(zip(grads, encoder_model.trainable_variables + decoder_model.trainable_variables))
        training_loss.append(np.mean(batch_losses))
        
        for test_english_batch, test_german_batch in test_dataset:
            test_decoder_input, test_decoder_output = prepare_decoder_data(test_english_batch, test_german_batch)
            test_loss_value, _ = forward_and_backward_prop(test_english_batch, test_decoder_input, test_decoder_output)
            validation_losses.append(test_loss_value.numpy())
        validation_loss.append(np.mean(validation_losses))  
        
        print(f"Epoch {epoch+1}/{epochs} - Training Loss: {training_loss[-1]:.4f}, Validation Loss: {validation_loss[-1]:.4f}")
        
  # Plot learning curves
    plt.figure(figsize=(8,5))
    plt.plot(range(1, epochs+1), training_loss, label="Training Loss")
    plt.plot(range(1, epochs+1), validation_loss, label="Validation Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Learning Curves")
    plt.legend()
    plt.show()
    
    return training_loss, validation_loss

In [ ]:
training_loop(train_dataset, validation_dataset, epochs=5)